In [ ]:
# 1. Load the committed weights and the results saved by train.ipynb (nothing is trained here)
import json
import os
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ultralytics import YOLO

ART = Path("artifacts")
summary = json.loads((ART / "metrics.json").read_text())
scores = pd.read_csv(ART / "scores.csv")
sweep = pd.read_csv(ART / "cost_sweep.csv")
per_class = pd.read_csv(ART / "per_class.csv")
staffing = pd.read_csv(ART / "staffing.csv")
model = YOLO("weights/fruit_yolo11n_cls.pt")

CLASSES, ROTTEN = summary["classes"], summary["rotten_classes"]
A = summary["assumptions"]
COST, BASE_RATE, N = A["cost"], A["base_rate"], A["fruit_per_day"]
THRESHOLD, BAND = summary["threshold"], summary["band"]
SETS = {"External photos: check half": "external_check", "External photos: tuning half": "external_tune",
        "Test split": "test", "Validation split": "valid"}

In [ ]:
# 2. Cost of a pass / review / fail rule per day (same formula as in training)
def rule_cost(d, low, high):
    r, f = d.p_rotten[d.is_rotten == 1].to_numpy(), d.p_rotten[d.is_rotten == 0].to_numpy()
    r_pass, r_fail = (r < low).mean(), (r >= high).mean()
    f_pass, f_fail = (f < low).mean(), (f >= high).mean()
    r_review, f_review = 1 - r_pass - r_fail, 1 - f_pass - f_fail
    error = (BASE_RATE * (r_pass * COST["missed_rotten"] + (r_review + r_fail) * COST["caught_rotten"])
             + (1 - BASE_RATE) * (f_fail * COST["discarded_good"] + (f_pass + f_review) * COST["passed_good"]))
    share = BASE_RATE * r_review + (1 - BASE_RATE) * f_review
    people = int(np.ceil(round(share * N / A["review_capacity_per_person"], 6)))
    per_day = N * error + people * A["shift_hours"] * A["wage_per_hour"]
    return {"€ per 1,000 fruit": round(1000 * per_day / N, 2), "€ per day": round(per_day),
            "share reviewed": f"{share:.1%}", "reviewers": people,
            "rotten fruit packed / day": round(N * BASE_RATE * r_pass),
            "good fruit thrown away / day": round(N * (1 - BASE_RATE) * f_fail)}


def decide(p, low, high):
    return "pass" if p < low else ("human review" if p < high else "fail")

In [ ]:
# 3. Upload a fruit: class probabilities and the decision at the current thresholds
def check_fruit(image, low, high):
    if image is None:
        return None, "Upload a photo of an apple, banana or orange."
    probs = model.predict(image, verbose=False)[0].probs.data.cpu().numpy()
    p_rotten = float(sum(q for c, q in zip(CLASSES, probs) if c in ROTTEN))
    return ({c: float(q) for c, q in zip(CLASSES, probs)},
            f"### Probability rotten: {p_rotten:.1%} → **{decide(p_rotten, low, high).upper()}**\n"
            f"(pass below {low:.2f}, human review from {low:.2f} to {high:.2f}, fail from {high:.2f})")

In [ ]:
# 4. Thresholds: confusion matrix, cost per 1,000 fruit, share sent for review, staff, cost curve
def threshold_view(set_label, low, high):
    low, high = min(low, high), max(low, high)
    d = scores[scores.split == SETS[set_label]]
    outcomes = ["pass", "human review", "fail"]
    table = pd.crosstab(d.is_rotten.map({1: "actually rotten", 0: "actually fresh"}),
                        d.p_rotten.apply(decide, args=(low, high)))
    table = table.reindex(index=["actually rotten", "actually fresh"], columns=outcomes, fill_value=0)
    matrix = go.Figure(go.Heatmap(z=table.values, x=outcomes, y=table.index, colorscale="Blues",
                                  showscale=False, text=table.values, texttemplate="%{text}"))
    matrix.update_layout(title=f"{set_label}: images by decision", height=320, template="plotly_white",
                         margin=dict(l=110, r=20, t=50, b=40))
    matrix.update_yaxes(autorange="reversed")

    mine = rule_cost(d, low, high)
    rules = {"Do nothing (pack everything)": (1.01, 1.01), "A person checks every fruit": (0.0, 1.01),
             f"Model alone (threshold {THRESHOLD})": (THRESHOLD, THRESHOLD),
             f"Chosen band ({BAND['low']} to {BAND['high']})": (BAND["low"], BAND["high"]),
             f"Your setting ({low:.2f} to {high:.2f})": (low, high)}
    policies = pd.DataFrame([{"policy": name, **rule_cost(d, lo, hi)} for name, (lo, hi) in rules.items()])
    text = (f"### Expected cost: €{mine['€ per 1,000 fruit']:,.2f} per 1,000 fruit (€{mine['€ per day']:,} a day)\n"
            f"**{mine['share reviewed']}** of fruit goes to a person → **{mine['reviewers']} reviewer(s)** "
            f"for a {N:,}-fruit day. Every day: {mine['rotten fruit packed / day']:,} rotten fruit packed, "
            f"{mine['good fruit thrown away / day']:,} good fruit thrown away.  \n"
            f"Assumes {BASE_RATE:.0%} of fruit is rotten, missed rotten €{COST['missed_rotten']}, "
            f"good fruit thrown away €{COST['discarded_good']}, {A['seconds_per_review']} s per check, "
            f"reviewers paid €{A['wage_per_hour']}/h for {A['shift_hours']} h shifts.")

    curve = go.Figure()
    for name, dash in [("external_check", "solid"), ("external_tune", "dash"), ("test", "dot"), ("validation", "dashdot")]:
        curve.add_trace(go.Scatter(x=sweep.threshold, y=sweep[f"cost_per_1000_{name}"], name=name.replace("_", " "),
                                   line=dict(dash=dash)))
    chosen = sweep.loc[np.isclose(sweep.threshold, THRESHOLD), "cost_per_1000_external_tune"].iloc[0]
    curve.add_trace(go.Scatter(x=[THRESHOLD], y=[chosen], mode="markers+text", name="chosen threshold",
                               marker=dict(size=12, color="red"), text=[f" {THRESHOLD}"], textposition="top right"))
    curve.add_vline(x=low, line_dash="dash", line_color="grey", annotation_text="pass | review")
    curve.add_vline(x=high, line_dash="dash", line_color="grey", annotation_text="review | fail",
                    annotation_position="top left")
    curve.update_layout(title="Cost per 1,000 fruit if the model decides alone", template="plotly_white",
                        xaxis_title="threshold on probability rotten", yaxis_title="€ per 1,000 fruit",
                        height=420, legend=dict(orientation="h", y=-0.25))
    return matrix, text, policies, curve

In [ ]:
# 5. Per-class results: test split for the model and both baselines, and the model on external photos
overall = pd.DataFrame(summary["overall"]).round(3)
per_class_table = per_class.rename(columns={"support": "images"}).round(3)
staffing_table = staffing.round(3)


def confusion_plot(model_name):
    z = np.array(summary["confusion"][model_name])
    fig = go.Figure(go.Heatmap(z=z, x=summary["pred_labels"], y=CLASSES, colorscale="Blues", showscale=False,
                               text=z, texttemplate="%{text}"))
    fig.update_layout(title=f"Confusion matrix: {model_name}", template="plotly_white", height=450,
                      xaxis_title="predicted", yaxis_title="actual")
    fig.update_yaxes(autorange="reversed")
    return fig

In [ ]:
# 6. Build the dashboard and launch it
with gr.Blocks(title="Rotten fruit check") as demo:
    gr.Markdown("# Rotten fruit check on a packing line\n"
                "A fine-tuned YOLO classifier gives each fruit a probability of being rotten. "
                "Below the first threshold it is packed, from the second it is thrown away, "
                "in between a worker checks it. Everything is loaded from `weights/` and `artifacts/`.")
    with gr.Row():
        low = gr.Slider(0, 1, value=BAND["low"], step=0.01, label="Pass below (probability rotten)")
        high = gr.Slider(0, 1, value=BAND["high"], step=0.01, label="Fail from (probability rotten)")

    with gr.Tab("Check a fruit"):
        with gr.Row():
            image = gr.Image(type="filepath", label="Photo of one fruit")
            with gr.Column():
                decision = gr.Markdown()
                probs = gr.Label(label="Class probabilities")
        for control in [image, low, high]:
            control.change(check_fruit, [image, low, high], [probs, decision])

    with gr.Tab("Thresholds & cost"):
        image_set = gr.Radio(list(SETS), value=list(SETS)[0], label="Images")
        cost_text = gr.Markdown()
        with gr.Row():
            matrix_plot = gr.Plot()
            curve_plot = gr.Plot()
        policy_table = gr.Dataframe(label="Your setting against the alternatives")
        gr.Dataframe(staffing_table, label="Best band for each number of reviewers "
                                           "(chosen on the tuning half, costed on the check half)")
        outputs = [matrix_plot, cost_text, policy_table, curve_plot]
        for control in [image_set, low, high]:
            control.change(threshold_view, [image_set, low, high], outputs)
        demo.load(threshold_view, [image_set, low, high], outputs)

    with gr.Tab("Per-class results"):
        gr.Markdown(f"Positive class = **rotten** ({', '.join(ROTTEN)}). Test split for the model and both "
                    "baselines, plus the model on photos from a second dataset it never saw.")
        gr.Dataframe(overall, label="Overall")
        gr.Dataframe(per_class_table, label="Per class")
        names = list(summary["confusion"])
        model_choice = gr.Dropdown(names, value=names[0], label="Model")
        confusion = gr.Plot(confusion_plot(names[0]))
        model_choice.change(confusion_plot, model_choice, confusion)

demo.launch(server_name=os.environ.get("GRADIO_SERVER_NAME", "127.0.0.1"), server_port=7860)